# Huấn luyện QKSR / RSIAT trên ImageNet-R hoặc ImageNet-A bằng Kaggle

Notebook này làm việc hoàn toàn với dữ liệu đã được gắn vào `/kaggle/input`:

- Mã nguồn được đọc từ **file ZIP hoặc thư mục repo** do bạn tự tải lên Kaggle. Kaggle có thể tự bung ZIP thành thư mục; notebook hỗ trợ cả hai dạng.
- Chọn `DATASET_NAME = 'imagenetr'` hoặc `'imageneta'`; notebook tự dùng đúng config gốc của repo.
- Dataset được tự động tìm trong các Kaggle Dataset đã thêm vào notebook.
- Notebook **không clone GitHub và không tải lại dataset**.
- Nếu dataset đã có `train/` và `test/`, notebook dùng trực tiếp hai thư mục đó. Nếu dataset chỉ có 200 thư mục lớp, notebook tạo split 80/20 bằng symlink trong `/kaggle/working`, không sao chép ảnh.

Trước khi chạy, hãy bật GPU trong phần cài đặt của Kaggle Notebook và thêm cả mã nguồn repo, pretrained model lẫn dataset cần chạy vào mục **Input**.

In [ ]:
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
PROJECT_DIR = WORK_ROOT / 'QG-RSA'
OUTPUT_ROOT = WORK_ROOT / 'qksr_runs'

# Chọn dataset tại đây: 'imageneta' hoặc 'imagenetr'.
DATASET_NAME = 'imageneta'
DATASET_SPECS = {
    'imagenetr': {
        'label': 'ImageNet-R',
        'folder': 'imagenet-r',
        'config': 'adapter_imagenetr.json',
        'aliases': ('imagenet-r', 'imagenetr', 'rendition'),
    },
    'imageneta': {
        'label': 'ImageNet-A',
        'folder': 'imagenet-a',
        'config': 'adapter_imageneta.json',
        'aliases': ('imagenet-a', 'imageneta', 'natural-adversarial'),
    },
}
if DATASET_NAME not in DATASET_SPECS:
    raise ValueError("DATASET_NAME phải là 'imageneta' hoặc 'imagenetr'.")
DATASET_SPEC = DATASET_SPECS[DATASET_NAME]
DATASET_LABEL = DATASET_SPEC['label']
DATASET_FOLDER = DATASET_SPEC['folder']
CONFIG_FILENAME = DATASET_SPEC['config']
DATASET_ALIASES = DATASET_SPEC['aliases']

# Để trống để notebook tự tìm. Có thể trỏ tới file ZIP, thư mục dataset, hoặc thư mục gốc của repo.
# Ví dụ: '/kaggle/input/datasets/thunvng/qg-rsa'
REPO_PATH = ''
# Để trống để tự tìm đúng dataset đã chọn; chỉ điền khi có nhiều ứng viên.
# Ví dụ ImageNet-A: '/kaggle/input/natural-adversarial-examples-imagenet-a/imagenet-a'
DATASET_PATH = ''
# Để trống để tự tìm model.safetensors trong Input.
PRETRAINED_MODEL_PATH = ''

# Tùy chọn: đường dẫn đến thư mục qksr_runs của một lần chạy Kaggle trước.
PREVIOUS_RUN_PATH = ''

RUN_PROFILE = 'smoke'       # 'smoke': kiểm tra nhanh; 'full': huấn luyện đầy đủ
USE_QKSR = True             # False để chạy RSIAT gốc
QKSR_PROFILE = 'legacy'     # 'current_margin' hoặc 'retention': ứng viên mới, cần chạy ablation
# 'retention' tự bật paired_eval, projector có dấu, relation KD và transport mean/cov; dùng được cho đối chứng USE_QKSR=False.
SSCA_FEATURE_MODE = 'legacy'  # 'paired_eval': ghép đúng ảnh trước/sau; thử riêng cho cả RSIAT và QKSR
SMOKE_TASKS = 2             # Hai task để kiểm tra cả loss cơ sở và loss tăng dần
SEED = 1993
TRAIN_FRACTION = 0.8        # Chỉ dùng khi dataset chưa có train/test

assert INPUT_ROOT.is_dir(), 'Không tìm thấy /kaggle/input. Hãy chạy notebook này trên Kaggle.'
assert RUN_PROFILE in {'smoke', 'full'}
assert QKSR_PROFILE in {'legacy', 'current_margin', 'retention'}
assert SSCA_FEATURE_MODE in {'legacy', 'paired_eval'}
assert 1 <= SMOKE_TASKS <= 10
assert 0.0 < TRAIN_FRACTION < 1.0
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
import subprocess
import torch

subprocess.run(['nvidia-smi'], check=False)
print('Phiên bản PyTorch:', torch.__version__)
print('CUDA khả dụng:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Chưa có GPU. Hãy bật GPU trong phần cài đặt Kaggle Notebook.')
print('GPU:', torch.cuda.get_device_name(0))

## 1. Tìm và giải nén mã nguồn

Cell này chỉ tìm trong `/kaggle/input`. Nguồn hợp lệ có thể là file ZIP hoặc thư mục đã được Kaggle giải nén; project được nhận diện bằng `main.py` và file config tương ứng với dataset đã chọn.

In [ ]:
import os
import shutil
import zipfile

def zip_chua_repo(zip_path):
    try:
        with zipfile.ZipFile(zip_path) as archive:
            names = {name.replace('\\', '/').strip('/') for name in archive.namelist()}
    except zipfile.BadZipFile:
        return False
    roots = {name[:-len('/main.py')] for name in names if name.endswith('/main.py')}
    if 'main.py' in names:
        roots.add('')
    return any(
        f'{root}/exps/{CONFIG_FILENAME}'.lstrip('/') in names
        for root in roots
    )

def tim_project_trong_thu_muc(root):
    root = Path(root)
    if not root.is_dir():
        return []
    candidates = set()
    if (root / 'main.py').is_file() and (root / 'exps' / CONFIG_FILENAME).is_file():
        candidates.add(root)
    for main_file in root.rglob('main.py'):
        project_root = main_file.parent
        if (project_root / 'exps' / CONFIG_FILENAME).is_file():
            candidates.add(project_root)
    return sorted(candidates, key=lambda path: (len(path.parts), str(path)))

if REPO_PATH:
    repo_source = Path(REPO_PATH)
    if not repo_source.exists():
        raise FileNotFoundError(f'Không tìm thấy nguồn repo: {repo_source}')
    if repo_source.is_file() and not zip_chua_repo(repo_source):
        raise ValueError(f'File không phải ZIP repo QG-RSA hợp lệ: {repo_source}')
    if repo_source.is_dir() and not tim_project_trong_thu_muc(repo_source):
        raise ValueError(f'Thư mục không chứa repo QG-RSA hợp lệ: {repo_source}')
else:
    mounted_projects = tim_project_trong_thu_muc(INPUT_ROOT)
    repo_zips = sorted(path for path in INPUT_ROOT.rglob('*.zip') if zip_chua_repo(path))
    if mounted_projects:
        # Kaggle thường tự giải nén file upload thành thư mục; ưu tiên dùng trực tiếp thư mục đó.
        if len(mounted_projects) > 1:
            raise RuntimeError(
                'Tìm thấy nhiều thư mục repo. Hãy điền REPO_PATH bằng một trong các đường dẫn:\n'
                + '\n'.join(map(str, mounted_projects))
            )
        repo_source = mounted_projects[0]
    elif repo_zips:
        if len(repo_zips) > 1:
            raise RuntimeError(
                'Tìm thấy nhiều ZIP repo. Hãy điền REPO_PATH bằng một trong các đường dẫn:\n'
                + '\n'.join(map(str, repo_zips))
            )
        repo_source = repo_zips[0]
    else:
        raise FileNotFoundError(
            'Không tìm thấy file ZIP hoặc thư mục repo trong /kaggle/input. '            'Hãy kiểm tra Input hoặc điền REPO_PATH.'
        )

if repo_source.is_file():
    extract_dir = WORK_ROOT / 'qg_rsa_source'
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True)
    extract_root = extract_dir.resolve()
    with zipfile.ZipFile(repo_source) as archive:
        for member in archive.infolist():
            destination = (extract_root / member.filename).resolve()
            if destination != extract_root and extract_root not in destination.parents:
                raise ValueError(f'Đường dẫn không an toàn trong ZIP: {member.filename}')
        archive.extractall(extract_root)
    project_candidates = tim_project_trong_thu_muc(extract_root)
else:
    project_candidates = tim_project_trong_thu_muc(repo_source)

if len(project_candidates) != 1:
    raise RuntimeError(f'Nguồn phải chứa đúng một project QG-RSA, tìm thấy {len(project_candidates)}.')
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
shutil.copytree(project_candidates[0], PROJECT_DIR)

if PREVIOUS_RUN_PATH:
    previous_run = Path(PREVIOUS_RUN_PATH)
    if not previous_run.is_dir():
        raise FileNotFoundError(f'Không tìm thấy lần chạy trước: {previous_run}')
    shutil.copytree(previous_run, OUTPUT_ROOT, dirs_exist_ok=True)
    print('Đã khôi phục checkpoint từ:', previous_run)

os.chdir(PROJECT_DIR)
print('Nguồn repo:', repo_source)
print('Thư mục project:', PROJECT_DIR)
print('Thư mục kết quả:', OUTPUT_ROOT)

In [ ]:
# Không cài lại torch/torchvision để giữ bộ CUDA tương thích có sẵn trên Kaggle.
%pip install -q --upgrade-strategy only-if-needed -r requirements-colab.txt

import timm
import torchvision
print('torchvision:', torchvision.__version__)
print('timm:', timm.__version__)

# Dùng pretrained model đã upload trong Input, không gọi Hugging Face.
if PRETRAINED_MODEL_PATH:
    pretrained_path = Path(PRETRAINED_MODEL_PATH)
    if not pretrained_path.is_file():
        raise FileNotFoundError(f'Không tìm thấy pretrained model: {pretrained_path}')
else:
    model_candidates = sorted(INPUT_ROOT.rglob('model.safetensors'))
    if len(model_candidates) != 1:
        raise RuntimeError(
            f'Cần đúng một model.safetensors trong Input, tìm thấy {len(model_candidates)}. '
            'Hãy điền PRETRAINED_MODEL_PATH nếu có nhiều file.'
        )
    pretrained_path = model_candidates[0]

from safetensors import safe_open
with safe_open(str(pretrained_path), framework='pt', device='cpu') as checkpoint:
    pretrained_keys = set(checkpoint.keys())
required_keys = {'patch_embed.proj.weight', 'blocks.0.attn.qkv.weight'}
missing_keys = required_keys - pretrained_keys
if missing_keys:
    raise ValueError(f'Checkpoint không đúng ViT-B/16 của timm; thiếu key: {sorted(missing_keys)}')

adapter_source = PROJECT_DIR / 'network' / 'vision_transformer_adapter.py'
adapter_text = adapter_source.read_text(encoding='utf-8')
online_line = '    checkpoint_model = timm.create_model("vit_base_patch16_224_in21k", pretrained=True, num_classes=0)'
local_block = f'''    checkpoint_model = timm.create_model(
        "vit_base_patch16_224.augreg_in21k", pretrained=False, num_classes=0
    )
    from safetensors.torch import load_file
    local_state_dict = load_file({str(pretrained_path)!r})
    local_state_dict.pop("head.weight", None)
    local_state_dict.pop("head.bias", None)
    local_load = checkpoint_model.load_state_dict(local_state_dict, strict=False)
    if local_load.missing_keys or local_load.unexpected_keys:
        raise ValueError(f'Checkpoint local không khớp backbone: {{local_load}}')'''
if online_line in adapter_text:
    adapter_source.write_text(adapter_text.replace(online_line, local_block, 1), encoding='utf-8')
    print('Đã cấu hình pretrained model local:', pretrained_path)
elif 'local_state_dict = load_file(' in adapter_text:
    print('Source đã dùng pretrained model local.')
else:
    raise RuntimeError('Không tìm thấy đoạn nạp pretrained model cần thay trong source.')

subprocess.run([
    'python', '-m', 'py_compile', 'main.py', 'trainer.py',
    'models/base.py', 'models/RSIAT_adapter.py',
    'utils/quantum_kernel.py', 'utils/qksr_statistics.py'
], check=True)
print('Kiểm tra cú pháp mã nguồn: đạt.')

## 2. Tìm và chuẩn bị dataset đã chọn

Loader của repo cần `data/datasets/<dataset>/train/<lớp>` và `test/<lớp>`. Cell chỉ tìm dữ liệu trong `/kaggle/input`; nó không gọi API và không tải dữ liệu từ Internet.

In [ ]:
import random

IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def thu_muc_lop(root):
    root = Path(root)
    if not root.is_dir():
        return []
    result = []
    for directory in root.iterdir():
        if directory.is_dir() and any(
            file.is_file() and file.suffix.lower() in IMAGE_SUFFIXES
            for file in directory.iterdir()
        ):
            result.append(directory)
    return sorted(result)

def loai_cau_truc(root):
    root = Path(root)
    train_classes = thu_muc_lop(root / 'train')
    test_classes = thu_muc_lop(root / 'test')
    if len(train_classes) == 200 and len(test_classes) == 200:
        if [p.name for p in train_classes] == [p.name for p in test_classes]:
            return 'đã chia train/test'
    if len(thu_muc_lop(root)) == 200:
        return 'chưa chia'
    return None

if DATASET_PATH:
    source_root = Path(DATASET_PATH)
    if not source_root.is_relative_to(INPUT_ROOT):
        raise ValueError('DATASET_PATH phải nằm trong /kaggle/input.')
    if loai_cau_truc(source_root) is None:
        raise ValueError(f'Không nhận ra cấu trúc {DATASET_LABEL} tại {source_root}')
else:
    split_candidates = []
    flat_candidates = []
    for directory in INPUT_ROOT.rglob('*'):
        if not directory.is_dir():
            continue
        normalized_path = str(directory).lower().replace('_', '-')
        if not any(alias in normalized_path for alias in DATASET_ALIASES):
            continue
        detected_layout = loai_cau_truc(directory)
        if detected_layout == 'đã chia train/test':
            split_candidates.append(directory)
        elif detected_layout == 'chưa chia':
            flat_candidates.append(directory)
    # Ưu tiên root có train/test; không chọn nhầm chính thư mục train hoặc test như một dataset phẳng.
    data_candidates = split_candidates if split_candidates else flat_candidates
    data_candidates = sorted(set(data_candidates), key=lambda p: (len(p.parts), str(p)))
    if not data_candidates:
        raise FileNotFoundError(
            f'Không tìm thấy {DATASET_LABEL} gồm 200 lớp trong /kaggle/input. '            'Hãy kiểm tra dataset đã được thêm vào Input hoặc điền DATASET_PATH.'
        )
    if len(data_candidates) > 1:
        raise RuntimeError(
            f'Tìm thấy nhiều thư mục có cấu trúc {DATASET_LABEL}. '            'Hãy điền DATASET_PATH bằng một trong các đường dẫn:\n'
            + '\n'.join(map(str, data_candidates))
        )
    source_root = data_candidates[0]

layout = loai_cau_truc(source_root)
print(f'{DATASET_LABEL}:', source_root)
print('Cấu trúc:', layout)

if layout == 'đã chia train/test':
    prepared_root = source_root
else:
    prepared_root = WORK_ROOT / f'{DATASET_FOLDER}-prepared'
    if prepared_root.exists():
        shutil.rmtree(prepared_root)
    rng = random.Random(SEED)
    for class_dir in thu_muc_lop(source_root):
        images = sorted(
            path for path in class_dir.iterdir()
            if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
        )
        if len(images) < 2:
            raise ValueError(f'Lớp {class_dir.name} có ít hơn 2 ảnh.')
        rng.shuffle(images)
        cut = min(max(1, int(len(images) * TRAIN_FRACTION)), len(images) - 1)
        for split, split_images in [('train', images[:cut]), ('test', images[cut:])]:
            destination = prepared_root / split / class_dir.name
            destination.mkdir(parents=True, exist_ok=True)
            for image in split_images:
                (destination / image.name).symlink_to(image.resolve())

train_classes = thu_muc_lop(prepared_root / 'train')
test_classes = thu_muc_lop(prepared_root / 'test')
if [p.name for p in train_classes] != [p.name for p in test_classes]:
    raise ValueError('Danh sách lớp của train và test không giống nhau.')
if len(train_classes) != 200:
    raise ValueError(f'{DATASET_LABEL} cần 200 lớp, nhưng tìm thấy {len(train_classes)} lớp.')

repo_data = PROJECT_DIR / 'data' / 'datasets' / DATASET_FOLDER
repo_data.parent.mkdir(parents=True, exist_ok=True)
if repo_data.is_symlink() or repo_data.is_file():
    repo_data.unlink()
elif repo_data.exists():
    shutil.rmtree(repo_data)
repo_data.symlink_to(prepared_root.resolve(), target_is_directory=True)

train_count = sum(sum(1 for p in directory.iterdir() if p.is_file()) for directory in train_classes)
test_count = sum(sum(1 for p in directory.iterdir() if p.is_file()) for directory in test_classes)
print(f'Số lớp: {len(train_classes)} | train: {train_count:,} ảnh | test: {test_count:,} ảnh')
print('Liên kết dữ liệu của repo:', repo_data, '->', repo_data.resolve())

## 3. Tạo cấu hình huấn luyện

Cấu hình giữ nguyên các siêu tham số riêng của dataset từ file tương ứng trong `exps/`. `QKSR_PROFILE='legacy'` giữ objective cũ; `'current_margin'` thử đổi đường gradient và giảm repulsion; `'retention'` thêm projector có dấu/khởi tạo identity, reset mỗi task, neo quan hệ cosine và transport cả mean/covariance. Retention tự bật `paired_eval`, giữ tất cả checkpoint và có đối chứng `USE_QKSR=False` dùng cùng các sửa chung. Đây đều là ứng viên chưa được chứng minh tăng accuracy. Smoke chạy hai task, mỗi task 2 epoch, CA tắt: không đủ kiểm tra hiệu quả replay; muốn kiểm tra CA hãy đặt `config['ca_epochs']=1` trước khi lưu cấu hình. Dùng `val_ratio>0` để chọn tham số trên held-out của toàn bộ lớp đã thấy; cấu hình mặc định `val_ratio=0` đánh giá test, không dùng để tuning. Full chỉ resume checkpoint tương thích trong cùng profile. Cần upload lại ZIP source mới, mở session mới và không resume checkpoint legacy khi đổi objective.

In [ ]:
import json

source_config = PROJECT_DIR / 'exps' / CONFIG_FILENAME
config = json.loads(source_config.read_text(encoding='utf-8'))
config.update({
    'prefix': f'{DATASET_NAME}_qksr' if USE_QKSR else f'{DATASET_NAME}_rsiat',
    'dataset': DATASET_NAME,
    'seed': [SEED],
    'device': ['0'],
    'output_root': str(OUTPUT_ROOT),
    'resume': RUN_PROFILE == 'full',
    'val_ratio': 0.0,
    'num_workers': 2,
    'stats_num_workers': 2,
    'pin_memory': True,
    'persistent_workers': True,
    'use_quantum_kernel_base': USE_QKSR,
    'use_quantum_kernel_inc': USE_QKSR,
    'ssca_feature_mode': SSCA_FEATURE_MODE,
})

if USE_QKSR:
    config.update({
        'q_kernel_type': 'pqk',
        'q_kernel_order': 1,
        'q_order2_weight': 1.0,
        'q_num_qubits': 8,
        'q_num_layers': 2,
        'q_reupload': False,
        'q_dtype': 'float32',
        'q_gamma_mode': 'bounded_learned',
        'q_calib_samples': 512,
        'q_init_seed': 1234,
        'q_metric_lr_mult': 0.1,
        'q_inc_train_mode': 'frozen',
        'inc_loss_mode': 'mean',
        'q_inc_pair': 'old_proj',
        'rs_margin_q': 0.5,
        'rs_margin_q_mode': 'fixed',
        'rs_margin_quantile': 0.9,
        'rs_margin_inc': 0.3,
        'q_inc_weight': 1.0,
        'q_inc_warmup_epochs': 0,
    })
    if QKSR_PROFILE == 'current_margin':
        config.update({'q_inc_pair': 'current', 'inc_loss_mode': 'margin',
                       'q_inc_weight': 0.25, 'q_inc_warmup_epochs': 3})
        config['prefix'] += '_current_margin'

if QKSR_PROFILE == 'retention':
    try:
        from utils.qksr_profiles import RETENTION_OPTIONS, RETENTION_SHARED_OPTIONS
    except ImportError as error:
        raise RuntimeError('Repo trong Input chưa có profile retention. Hãy upload ZIP source mới và mở session mới.') from error
    config.update(RETENTION_OPTIONS if USE_QKSR else RETENTION_SHARED_OPTIONS)
    config['prefix'] += '_retention'

if config['ssca_feature_mode'] != 'legacy':
    config['prefix'] += '_' + config['ssca_feature_mode']

if RUN_PROFILE == 'smoke':
    config.update({
        'prefix': config['prefix'] + '_smoke',
        'resume': False,
        'max_tasks_per_run': SMOKE_TASKS,
        'init_epochs': 2,
        'inc_epochs': 2,
        'ca_epochs': 0,
        'batch_size': min(32, int(config['batch_size'])),
    })
else:
    config.pop('max_tasks_per_run', None)

CONFIG_PATH = WORK_ROOT / f'{DATASET_NAME}_kaggle.json'
CONFIG_PATH.write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding='utf-8')
print('Cấu hình đã tạo:', CONFIG_PATH)
print(json.dumps(config, indent=2, ensure_ascii=False))

## 4. Kiểm tra trước khi huấn luyện

In [ ]:
import sys
subprocess.run([
    sys.executable, '-m', 'unittest', 'discover',
    '-s', str(PROJECT_DIR / 'tests'),
    '-p', 'test_*.py', '-q',
], cwd=PROJECT_DIR, check=True)

from data.data_manager import DataManager
from models.RSIAT_adapter import Learner
if QKSR_PROFILE == 'retention' and not hasattr(Learner, '_transport_old_statistics'):
    raise RuntimeError('Repo trong Input chưa có bản retention. Hãy upload ZIP source mới và mở session mới.')
if (USE_QKSR and QKSR_PROFILE != 'legacy') or SSCA_FEATURE_MODE != 'legacy':
    if not hasattr(Learner, '_quantum_inc_weight') or not hasattr(Learner, '_build_ssca_loader'):
        raise RuntimeError('Repo trong Input chưa có cải tiến QKSR. Hãy upload ZIP source mới và chạy lại từ đầu.')
probe = DataManager(DATASET_NAME, True, SEED, config['init_cls'], config['increment'])
if probe.nb_tasks != 10:
    raise RuntimeError(f'Với 200 lớp và bước tăng 20, phải có 10 task; nhận được {probe.nb_tasks}.')
print('Số task:', probe.nb_tasks)
print('20 nhãn đầu trong thứ tự lớp:', probe._class_order[:20])
print('Toàn bộ kiểm tra trước khi huấn luyện đã đạt.')

## 5. Bắt đầu huấn luyện

Nên chạy `RUN_PROFILE = 'smoke'` trước. Khi smoke hoàn tất, đổi thành `'full'`, chạy lại cell tạo cấu hình, cell kiểm tra và cell huấn luyện.

In [ ]:
import sys
command = [sys.executable, '-u', 'main.py', '--config', str(CONFIG_PATH)]
print('Lệnh đang chạy:', ' '.join(command))
subprocess.run(command, cwd=PROJECT_DIR, check=True)

## 6. Kiểm tra và đóng gói kết quả

Cell cuối đóng gói cấu hình và log thành ZIP nhỏ. Checkpoint được giữ nguyên trong `/kaggle/working/qksr_runs` để bạn lưu thành Kaggle Dataset cho lần chạy tiếp theo.

In [ ]:
current_log_root = OUTPUT_ROOT / 'logs' / 'adapter' / DATASET_NAME
current_checkpoint_root = OUTPUT_ROOT / 'ckpt' / config['prefix'] / DATASET_NAME
logs = sorted(current_log_root.rglob('*.log'), key=lambda path: path.stat().st_mtime)
checkpoints = sorted(current_checkpoint_root.rglob('task_*.pkl'), key=lambda path: path.stat().st_mtime)
print('Log mới nhất:', logs[-1] if logs else 'chưa có')
print('Checkpoint mới nhất:', checkpoints[-1] if checkpoints else 'chưa có')

bundle = WORK_ROOT / f'{DATASET_NAME}_run_metadata'
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir()
shutil.copy2(CONFIG_PATH, bundle / CONFIG_PATH.name)
for index, log in enumerate(logs):
    shutil.copy2(log, bundle / f'{index:02d}_{log.name}')
archive = shutil.make_archive(str(WORK_ROOT / f'{DATASET_NAME}_run_metadata'), 'zip', bundle)
print('ZIP metadata:', archive)
print('Thư mục checkpoint cần lưu:', OUTPUT_ROOT)